# Entrenamiento del modelo – Práctica MLOps

Este notebook forma parte del flujo **Colab → GitHub → procesar_datos.py → entrenamiento.ipynb → modelo.joblib → Gradio / FastAPI / Dashboard**.

Objetivo: predecir la columna `resultado` (**SI / NO**) a partir de variables agroclimáticas
(`temperatura`, `humedad`, `radiacion_solar`, `precipitacion`, `viento`), el tipo de `cultivo` y la
`ciudad` asociada a cada cultivo (**cafe → Chanchamayo, papa → Huancayo, uva → Ica**).

> **Sobre `ciudad`:** es una variable predictora categórica que entra al pipeline con one-hot encoding igual que
> `cultivo`. Como en este dataset cada cultivo se asocia a una única ciudad, ambas variables son colineales: el modelo
> reparte su efecto conjunto entre los coeficientes `cultivo_*` y `ciudad_*`, que deben leerse como un par.

> **Nota:** como la variable objetivo es binaria (SI/NO) se usa **`LogisticRegression`** (modelo lineal de clasificación)
> en lugar de `LinearRegression`, que es para variables continuas. Así el modelo devuelve probabilidades y permite
> construir la **matriz de confusión** y las métricas que consume el dashboard.

Artefactos generados:

| Artefacto | Descripción |
|-----------|-------------|
| `models/modelo.joblib` | Pipeline completo (pre-procesamiento + modelo) serializado |
| `models/metricas.json` | Métricas, matriz de confusión, curva ROC, coeficientes y matriz de correlación |
| `output/matriz_confusion.png` | Gráfico de la matriz de confusión (hold-out) |
| `output/heatmap_correlacion.png` | Mapa de calor de correlaciones |


In [1]:
# ── 1. Librerías y rutas ─────────────────────────────────────────────────────
import json
from datetime import datetime, timezone
from pathlib import Path

import joblib
import matplotlib
matplotlib.use("Agg")  # backend sin pantalla (Colab / CI)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score, roc_auc_score, roc_curve)
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# El notebook puede ejecutarse desde /notebooks (local) o desde la raíz del proyecto (Colab)
CWD = Path.cwd()
BASE_DIR = CWD.parent if CWD.name == "notebooks" else CWD
DATA_PATH = BASE_DIR / "data" / "datos.csv"
MODELS_DIR = BASE_DIR / "models"
OUTPUT_DIR = BASE_DIR / "output"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURES_NUM = ["temperatura", "humedad", "radiacion_solar", "precipitacion", "viento"]
FEATURE_CAT = "cultivo"
FEATURE_CIUDAD = "ciudad"                       # variable predictora asociada al cultivo
CIUDAD_POR_CULTIVO = {"cafe": "chanchamayo", "papa": "huancayo", "uva": "ica"}
FEATURES_CAT = [FEATURE_CAT, FEATURE_CIUDAD]
TARGET = "resultado"
RANDOM_STATE = 42

print("Directorio base :", BASE_DIR)
print("Dataset         :", DATA_PATH, "| existe:", DATA_PATH.exists())
print("scikit-learn    :", sklearn.__version__)

Directorio base : /home/user/Practica_MLOPS
Dataset         : /home/user/Practica_MLOPS/data/datos.csv | existe: True
scikit-learn    : 1.6.1


In [2]:
# ── 2. Carga y exploración rápida ────────────────────────────────────────────
df = pd.read_csv(DATA_PATH)
df[TARGET] = df[TARGET].astype(str).str.strip().str.upper()
df[FEATURE_CAT] = df[FEATURE_CAT].astype(str).str.strip().str.lower()
# `ciudad` es una variable predictora asociada 1:1 al cultivo (cafe→chanchamayo, papa→huancayo, uva→ica).
# Si el CSV no la trae, se deduce del cultivo; si la trae, se normaliza.
if FEATURE_CIUDAD not in df.columns:
    df.insert(df.columns.get_loc(FEATURE_CAT) + 1, FEATURE_CIUDAD, df[FEATURE_CAT].map(CIUDAD_POR_CULTIVO))
df[FEATURE_CIUDAD] = df[FEATURE_CIUDAD].astype(str).str.strip().str.lower()
df = df.dropna(subset=FEATURES_NUM + FEATURES_CAT + [TARGET]).reset_index(drop=True)

print("Forma:", df.shape)
display(df.head())
print("\nDistribución del objetivo:")
print(df[TARGET].value_counts())
print("\nTasa de SI por cultivo:")
print(df.groupby(FEATURE_CAT)[TARGET].apply(lambda s: (s == "SI").mean()).round(3))
print("\nCiudad asociada a cada cultivo:")
print(df.groupby([FEATURE_CAT, FEATURE_CIUDAD]).size())
display(df[FEATURES_NUM].describe().round(2))

Forma: (1000, 9)


,id,cultivo,ciudad,temperatura,humedad,radiacion_solar,precipitacion,viento,resultado
0,1,uva,ica,19.6,66.0,18.7,1.1,2.4,NO
1,2,uva,ica,30.5,43.0,21.3,0.2,4.1,NO
2,3,uva,ica,26.4,59.6,25.7,5.6,6.9,SI
3,4,cafe,chanchamayo,22.7,89.3,8.1,15.1,7.1,NO
4,5,cafe,chanchamayo,20.2,71.5,18.7,4.5,4.1,SI



Distribución del objetivo:
resultado
NO    812
SI    188
Name: count, dtype: int64

Tasa de SI por cultivo:
cultivo
cafe    0.354
papa    0.055
uva     0.208
Name: resultado, dtype: float64

Ciudad asociada a cada cultivo:
cultivo  ciudad     
cafe     chanchamayo    260
papa     huancayo       380
uva      ica            360
dtype: int64


,temperatura,humedad,radiacion_solar,precipitacion,viento
count,1000.00,1000.00,1000.00,1000.0,1000.00
mean,20.67,68.69,18.53,7.7,6.80
std,4.19,14.38,4.61,4.9,3.28
min,7.50,24.70,3.20,0.0,0.00
25%,17.80,58.70,15.40,4.2,4.60
50%,20.60,70.05,18.20,7.0,6.60
75%,23.40,80.00,21.80,11.0,8.80
max,33.50,100.00,30.90,26.0,19.50


In [3]:
# ── 3. Mapa de calor de correlaciones ────────────────────────────────────────
df_corr = df[FEATURES_NUM].copy()
df_corr["resultado_bin"] = (df[TARGET] == "SI").astype(int)
matriz_corr = df_corr.corr().round(3)

plt.figure(figsize=(7, 5.5))
sns.heatmap(matriz_corr, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1, square=True)
plt.title("Matriz de correlación (Pearson)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "heatmap_correlacion.png", dpi=120)
plt.show()
matriz_corr

,temperatura,humedad,radiacion_solar,precipitacion,viento,resultado_bin
temperatura,1.000,-0.423,0.306,-0.225,-0.043,0.238
humedad,-0.423,1.000,-0.374,0.463,-0.138,0.059
radiacion_solar,0.306,-0.374,1.000,-0.395,-0.013,0.082
precipitacion,-0.225,0.463,-0.395,1.000,-0.061,0.059
viento,-0.043,-0.138,-0.013,-0.061,1.000,-0.055
resultado_bin,0.238,0.059,0.082,0.059,-0.055,1.000


In [4]:
# ── 4. División train / test y pipeline ──────────────────────────────────────
X = df[FEATURES_NUM + FEATURES_CAT]
y = (df[TARGET] == "SI").astype(int)          # 1 = SI, 0 = NO

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)

preprocesador = ColumnTransformer([
    ("num", StandardScaler(), FEATURES_NUM),
    ("cat", OneHotEncoder(handle_unknown="ignore"), FEATURES_CAT),   # cultivo + ciudad (one-hot)
])

modelo = Pipeline([
    ("pre", preprocesador),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, C=1.0, random_state=RANDOM_STATE)),
])

modelo.fit(X_train, y_train)
print(f"Entrenado con {len(X_train)} filas | test: {len(X_test)} filas | positivos en test: {int(y_test.sum())}")
modelo

Entrenado con 750 filas | test: 250 filas | positivos en test: 47


Pipeline(steps=[('pre',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['temperatura', 'humedad',
                                                   'radiacion_solar',
                                                   'precipitacion', 'viento']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['cultivo', 'ciudad'])])),
                ('clf',
                 LogisticRegression(class_weight='balanced', max_iter=1000,
                                    random_state=42))])

In [5]:
# ── 5. Evaluación en hold-out: métricas + matriz de confusión ────────────────
y_pred = modelo.predict(X_test)
y_proba = modelo.predict_proba(X_test)[:, 1]

cm_holdout = confusion_matrix(y_test, y_pred, labels=[0, 1])   # [[TN, FP], [FN, TP]]
metricas_holdout = {
    "accuracy":  round(float(accuracy_score(y_test, y_pred)), 4),
    "precision": round(float(precision_score(y_test, y_pred, zero_division=0)), 4),
    "recall":    round(float(recall_score(y_test, y_pred, zero_division=0)), 4),
    "f1":        round(float(f1_score(y_test, y_pred, zero_division=0)), 4),
    "roc_auc":   round(float(roc_auc_score(y_test, y_proba)), 4),
    "matriz_confusion": cm_holdout.tolist(),
    "n": int(len(y_test)),
}
print(json.dumps(metricas_holdout, indent=2))
print(classification_report(y_test, y_pred, target_names=["NO", "SI"], zero_division=0))

fig, ax = plt.subplots(figsize=(4.5, 4))
sns.heatmap(cm_holdout, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["NO", "SI"], yticklabels=["NO", "SI"], ax=ax)
ax.set_xlabel("Predicción"); ax.set_ylabel("Real"); ax.set_title("Matriz de confusión (hold-out)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "matriz_confusion.png", dpi=120)
plt.show()

{
  "accuracy": 0.668,
  "precision": 0.32,
  "recall": 0.6809,
  "f1": 0.4354,
  "roc_auc": 0.7246,
  "matriz_confusion": [
    [
      135,
      68
    ],
    [
      15,
      32
    ]
  ],
  "n": 250
}
              precision    recall  f1-score   support

          NO       0.90      0.67      0.76       203
          SI       0.32      0.68      0.44        47

    accuracy                           0.67       250
   macro avg       0.61      0.67      0.60       250
weighted avg       0.79      0.67      0.70       250



In [6]:
# ── 6. Validación cruzada estratificada (predicciones out-of-fold) ───────────
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
y_oof = cross_val_predict(modelo, X, y, cv=skf)
p_oof = cross_val_predict(modelo, X, y, cv=skf, method="predict_proba")[:, 1]

cm_cv = confusion_matrix(y, y_oof, labels=[0, 1])
metricas_cv = {
    "n_splits": 5,
    "accuracy":  round(float(accuracy_score(y, y_oof)), 4),
    "precision": round(float(precision_score(y, y_oof, zero_division=0)), 4),
    "recall":    round(float(recall_score(y, y_oof, zero_division=0)), 4),
    "f1":        round(float(f1_score(y, y_oof, zero_division=0)), 4),
    "roc_auc":   round(float(roc_auc_score(y, p_oof)), 4),
    "matriz_confusion": cm_cv.tolist(),
    "n": int(len(y)),
}
print(json.dumps(metricas_cv, indent=2))

fpr, tpr, _ = roc_curve(y, p_oof)
plt.figure(figsize=(4.5, 4))
plt.plot(fpr, tpr, label=f"ROC (AUC = {metricas_cv['roc_auc']:.2f})")
plt.plot([0, 1], [0, 1], "k--", lw=0.8)
plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title("Curva ROC (out-of-fold)"); plt.legend()
plt.tight_layout(); plt.show()

{
  "n_splits": 5,
  "accuracy": 0.663,
  "precision": 0.3295,
  "recall": 0.766,
  "f1": 0.4608,
  "roc_auc": 0.7506,
  "matriz_confusion": [
    [
      519,
      293
    ],
    [
      44,
      144
    ]
  ],
  "n": 1000
}


In [7]:
# ── 7. Importancia de variables (coeficientes del modelo lineal) ─────────────
# Nota: `ciudad` es perfectamente colineal con `cultivo` (relación 1:1), por lo que la regularización L2
# reparte el efecto conjunto entre cultivo_x y ciudad_y; deben interpretarse como un par (cultivo + ciudad).
nombres = modelo.named_steps["pre"].get_feature_names_out()
coefs = modelo.named_steps["clf"].coef_[0]
coeficientes = {str(n).replace("num__", "").replace("cat__", ""): round(float(c), 4)
                for n, c in zip(nombres, coefs)}
pd.Series(coeficientes).sort_values().plot(kind="barh", figsize=(6, 3.5), title="Coeficientes (log-odds de SI)")
plt.tight_layout(); plt.show()
coeficientes

{'temperatura': 0.7445,
 'humedad': 0.3741,
 'radiacion_solar': 0.3817,
 'precipitacion': 0.0073,
 'viento': 0.0771,
 'cultivo_cafe': 0.4153,
 'cultivo_papa': -0.4196,
 'cultivo_uva': -0.1693,
 'ciudad_chanchamayo': 0.4153,
 'ciudad_huancayo': -0.4196,
 'ciudad_ica': -0.1693}

In [8]:
# ── 8. Serialización: modelo.joblib + metricas.json ──────────────────────────
ruta_modelo = MODELS_DIR / "modelo.joblib"
joblib.dump(modelo, ruta_modelo)

metricas = {
    "modelo": "LogisticRegression",
    "version": "1.2.0",   # 1.1.0: variable `ciudad` · 1.2.0: reentrenado con el dataset ampliado (1000 filas)
    "fecha_entrenamiento": datetime.now(timezone.utc).astimezone().isoformat(timespec="seconds"),
    "sklearn_version": sklearn.__version__,
    "features_numericas": FEATURES_NUM,
    "feature_categorica": FEATURE_CAT,
    "feature_ciudad": FEATURE_CIUDAD,
    "features_categoricas": FEATURES_CAT,
    "categorias_cultivo": sorted(df[FEATURE_CAT].unique().tolist()),
    "categorias_ciudad": sorted(df[FEATURE_CIUDAD].unique().tolist()),
    "ciudad_por_cultivo": {c: str(v) for c, v in df.groupby(FEATURE_CAT)[FEATURE_CIUDAD].first().items()},
    "target": TARGET,
    "clases": ["NO", "SI"],
    "n_filas": int(len(df)),
    "n_train": int(len(X_train)),
    "n_test": int(len(X_test)),
    "holdout": metricas_holdout,
    "cv": metricas_cv,
    "roc_curve": {"fpr": [round(float(v), 4) for v in fpr], "tpr": [round(float(v), 4) for v in tpr]},
    "coeficientes": coeficientes,
    "intercepto": round(float(modelo.named_steps["clf"].intercept_[0]), 4),
    "medias_entrenamiento": {c: round(float(X_train[c].mean()), 4) for c in FEATURES_NUM},
    "desviaciones_entrenamiento": {c: round(float(X_train[c].std()), 4) for c in FEATURES_NUM},
    "rangos": {c: [round(float(df[c].min()), 2), round(float(df[c].max()), 2)] for c in FEATURES_NUM},
    "matriz_correlacion": {"columnas": matriz_corr.columns.tolist(), "valores": matriz_corr.values.tolist()},
}
with open(MODELS_DIR / "metricas.json", "w", encoding="utf-8") as f:
    json.dump(metricas, f, ensure_ascii=False, indent=2)

print("Modelo guardado en   :", ruta_modelo, f"({ruta_modelo.stat().st_size/1024:.1f} KB)")
print("Métricas guardadas en:", MODELS_DIR / "metricas.json")

Modelo guardado en   : /home/user/Practica_MLOPS/models/modelo.joblib (3.8 KB)
Métricas guardadas en: /home/user/Practica_MLOPS/models/metricas.json


In [9]:
# ── 9. Prueba de carga del artefacto (lo mismo que hacen Gradio y FastAPI) ───
modelo_cargado = joblib.load(ruta_modelo)
ejemplo = pd.DataFrame([{
    "temperatura": 21.0, "humedad": 80.0, "radiacion_solar": 16.0,
    "precipitacion": 12.0, "viento": 5.0, "cultivo": "cafe", "ciudad": "chanchamayo",
}])
prob_si = float(modelo_cargado.predict_proba(ejemplo)[0, 1])
print(f"Predicción: {'SI' if prob_si >= 0.5 else 'NO'}  (P(SI) = {prob_si:.3f})")

Predicción: SI  (P(SI) = 0.681)
